In [11]:
import argparse
import time
import torch
import torch.nn as nn

# Runtime metrics used by TransformerBlock.  Keep these global so the
# notebook can inspect FFN cost without requiring a profiler.
FFN_TIME_MS = []
FFN_MEM_BYTES = []

In [ ]:
#####################################
# Chapter 3
#####################################

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, dropout, num_heads, qkv_bias=False):
        super().__init__()

        # d_out phải chia hết cho số head
        # Ví dụ:
        # d_out = 768
        # num_heads = 12
        # => mỗi head có head_dim = 64
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads

        # Số chiều vector Q/K/V của MỖI head
        # Ví dụ: 768 / 12 = 64
        self.head_dim = d_out // num_heads

        # ============================================================
        # Q, K, V projection
        #
        # Input x:
        #   (batch, num_tokens, d_in)
        #
        # Sau Linear:
        #   Q/K/V:
        #   (batch, num_tokens, d_out)
        #
        # Ví dụ:
        # x = (1, 5, 768)
        # => Q/K/V = (1, 5, 768)
        # ============================================================
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Sau khi multi-head attention xong,
        # ghép tất cả head lại rồi chiếu thêm một Linear cuối.
        #
        # shape:
        # (batch, num_tokens, d_out)
        # ->
        # (batch, num_tokens, d_out)
        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)

        ####################################################
        # KV CACHE
        #
        # cache_k:
        #   lưu Key của các token đã xử lý trước đó
        #
        # cache_v:
        #   lưu Value của các token đã xử lý trước đó
        #
        # Shape cache:
        # (batch, total_cached_tokens, num_heads, head_dim)
        #
        # Ví dụ:
        # đã xử lý "I love AI"
        #
        # cache_k.shape = (1, 3, 12, 64)
        #
        # 1  = batch size
        # 3  = 3 token đã cache
        # 12 = số head
        # 64 = vector K của mỗi head
        ####################################################
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)

        # Vị trí hiện tại trong toàn sequence khi dùng cache.
        #
        # Ví dụ đã xử lý 3 token:
        # "I love AI"
        #
        # ptr_current_pos = 3
        #
        # token tiếp theo sẽ nằm ở position 3.
        self.ptr_current_pos = 0

    def forward(self, x, use_cache=False):

        # ============================================================
        # INPUT
        #
        # x.shape:
        # (batch, num_tokens, d_in)
        #
        # Không cache:
        # x có thể là cả sequence:
        #   [I, love, AI]
        #   shape = (1, 3, 768)
        #
        # Có cache khi decode:
        # x thường chỉ là token mới:
        #   [is]
        #   shape = (1, 1, 768)
        # ============================================================
        b, num_tokens, d_in = x.shape

        # ============================================================
        # TẠO K, V, Q MỚI
        #
        # Chú ý:
        # những tensor này chỉ được tính cho x HIỆN TẠI.
        #
        # Nếu x = toàn sequence:
        #   tính Q/K/V cho toàn sequence.
        #
        # Nếu x = chỉ token mới:
        #   chỉ tính Q/K/V cho token mới.
        #
        # Shape:
        # (b, num_tokens, d_in)
        # ->
        # (b, num_tokens, d_out)
        #
        # Ví dụ:
        # x.shape = (1, 1, 768)
        #
        # keys_new.shape   = (1, 1, 768)
        # values_new.shape = (1, 1, 768)
        # queries.shape    = (1, 1, 768)
        # ============================================================
        keys_new = self.W_key(x)
        values_new = self.W_value(x)
        queries = self.W_query(x)

        # ============================================================
        # SPLIT THÀNH NHIỀU HEAD
        #
        # Ban đầu:
        # (b, num_tokens, d_out)
        #
        # Ví dụ:
        # (1, 3, 768)
        #
        # d_out = num_heads * head_dim
        #       = 12 * 64
        #
        # Sau view:
        # (b, num_tokens, num_heads, head_dim)
        #
        # (1, 3, 12, 64)
        #
        # Nghĩa là:
        # mỗi token có 12 head,
        # mỗi head sở hữu vector 64 chiều.
        # ============================================================
        keys_new = keys_new.view(
            b,
            num_tokens,
            self.num_heads,
            self.head_dim
        )

        values_new = values_new.view(
            b,
            num_tokens,
            self.num_heads,
            self.head_dim
        )

        queries = queries.view(
            b,
            num_tokens,
            self.num_heads,
            self.head_dim
        )

        ####################################################
        # KV CACHE
        ####################################################

        if use_cache:

            # ========================================================
            # Nếu chưa có cache:
            #
            # Ví dụ lần đầu prefill:
            #
            # x = [I, love, AI]
            #
            # keys_new:
            # [K_I, K_love, K_AI]
            #
            # values_new:
            # [V_I, V_love, V_AI]
            #
            # Ta lưu chúng lại.
            # ========================================================
            if self.cache_k is None:
                self.cache_k, self.cache_v = keys_new, values_new

            else:
                # ====================================================
                # Nếu cache đã tồn tại:
                #
                # Ví dụ cache cũ:
                #
                # cache_k =
                # [K_I, K_love, K_AI]
                #
                # token mới x = [is]
                #
                # keys_new =
                # [K_is]
                #
                # torch.cat(..., dim=1)
                #
                # => cache mới:
                #
                # [K_I, K_love, K_AI, K_is]
                #
                # dim=1 chính là chiều TOKEN.
                #
                # Shape:
                #
                # cache cũ:
                # (1, 3, num_heads, head_dim)
                #
                # keys_new:
                # (1, 1, num_heads, head_dim)
                #
                # cache mới:
                # (1, 4, num_heads, head_dim)
                # ====================================================
                self.cache_k = torch.cat(
                    [self.cache_k, keys_new],
                    dim=1
                )

                self.cache_v = torch.cat(
                    [self.cache_v, values_new],
                    dim=1
                )

            # ========================================================
            # Query:
            # chỉ query của x hiện tại.
            #
            # Nhưng K/V:
            # dùng TOÀN BỘ lịch sử từ cache.
            #
            # Ví dụ đang xử lý "is":
            #
            # Q:
            #   Q_is
            #
            # K:
            #   K_I
            #   K_love
            #   K_AI
            #   K_is
            #
            # V:
            #   V_I
            #   V_love
            #   V_AI
            #   V_is
            # ========================================================
            keys, values = self.cache_k, self.cache_v

        else:
            # ========================================================
            # KHÔNG DÙNG CACHE
            #
            # keys/values chỉ là K/V vừa tính từ x hiện tại.
            #
            # Ví dụ:
            # x = [I, love, AI]
            #
            # keys =
            # [K_I, K_love, K_AI]
            #
            # Không lưu lại cho lần gọi tiếp theo.
            # ========================================================
            keys, values = keys_new, values_new

            # Forward không cache được xem là request mới.
            # Xóa cache cũ để request trước không ảnh hưởng request mới.
            self.cache_k, self.cache_v = None, None
            self.ptr_current_pos = 0

        ####################################################

        # ============================================================
        # ĐỔI THỨ TỰ DIM ĐỂ TÍNH ATTENTION
        #
        # Trước:
        #
        # keys:
        # (b, num_tokens_K, num_heads, head_dim)
        #
        # queries:
        # (b, num_tokens_Q, num_heads, head_dim)
        #
        # Sau transpose(1, 2):
        #
        # (b, num_heads, num_tokens, head_dim)
        #
        # Ví dụ cache có 4 token:
        #
        # keys:
        # (1, 4, 12, 64)
        #
        # ->
        #
        # (1, 12, 4, 64)
        #
        # Mục đích:
        # mỗi head tự attention độc lập.
        # ============================================================
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # ============================================================
        # ATTENTION SCORE = Q @ K^T
        #
        # queries:
        # (b, heads, num_tokens_Q, head_dim)
        #
        # keys.transpose(2,3):
        # (b, heads, head_dim, num_tokens_K)
        #
        # =>
        #
        # attn_scores:
        # (b, heads, num_tokens_Q, num_tokens_K)
        #
        #
        # Ví dụ đang decode "is":
        #
        # queries:
        # (1, 12, 1, 64)
        #
        # keys:
        # (1, 12, 4, 64)
        #
        # result:
        # (1, 12, 1, 4)
        #
        # Nghĩa là mỗi head tạo 4 score:
        #
        # Q_is · K_I
        # Q_is · K_love
        # Q_is · K_AI
        # Q_is · K_is
        # ============================================================
        attn_scores = queries @ keys.transpose(2, 3)

        ####################################################
        # CAUSAL MASK
        ####################################################

        # ============================================================
        # Số Query token hiện tại
        #
        # Không cache:
        # có thể Q có 5 token.
        #
        # Decode cache:
        # thường Q chỉ có 1 token.
        # ============================================================
        num_tokens_Q = queries.shape[-2]

        # ============================================================
        # Số Key hiện tại.
        #
        # Nếu dùng cache:
        # đây là toàn bộ token đã cache.
        #
        # Ví dụ:
        #
        # Q chỉ có "is"       => num_tokens_Q = 1
        #
        # K có:
        # I love AI is
        #
        # => num_tokens_K = 4
        # ============================================================
        num_tokens_K = keys.shape[-2]

        device = queries.device

        if use_cache:
            # ========================================================
            # Xác định POSITION THẬT của Query trong toàn sequence.
            #
            # Ví dụ cache trước đó đã xử lý:
            #
            # I     love     AI
            # 0      1       2
            #
            # ptr_current_pos = 3
            #
            # token mới:
            # is
            #
            # sẽ có position = 3.
            #
            #
            # Nếu num_tokens_Q = 1:
            #
            # arange(3, 4)
            #
            # =>
            #
            # q_positions = [3]
            # ========================================================
            q_positions = torch.arange(
                self.ptr_current_pos,
                self.ptr_current_pos + num_tokens_Q,
                device=device,
                dtype=torch.long,
            )

            # Sau khi xử lý xong Query hiện tại,
            # con trỏ tiến lên.
            #
            # Ví dụ:
            # trước = 3
            # num_tokens_Q = 1
            #
            # sau = 4
            self.ptr_current_pos += num_tokens_Q

        else:
            # ========================================================
            # Không cache:
            #
            # x chính là sequence hiện tại,
            # nên position đơn giản là:
            #
            # [0, 1, 2, ..., num_tokens_Q-1]
            #
            # Ví dụ:
            # "I love AI"
            #
            # q_positions = [0, 1, 2]
            # ========================================================
            q_positions = torch.arange(
                num_tokens_Q,
                device=device,
                dtype=torch.long
            )

            self.ptr_current_pos = 0

        # ============================================================
        # Position của toàn bộ Key.
        #
        # Ví dụ K cache:
        #
        # I     love     AI     is
        # 0      1       2      3
        #
        # =>
        #
        # k_positions = [0, 1, 2, 3]
        # ============================================================
        k_positions = torch.arange(
            num_tokens_K,
            device=device,
            dtype=torch.long
        )

        # ============================================================
        # TẠO CAUSAL MASK
        #
        # Rule:
        #
        # Nếu:
        #
        # q_position < k_position
        #
        # nghĩa là K nằm trong TƯƠNG LAI của Q
        # => phải che.
        #
        #
        # Ví dụ full sequence:
        #
        # q_positions = [0,1,2]
        # k_positions = [0,1,2]
        #
        #
        # mask:
        #
        #           K0      K1      K2
        #
        # Q0       False   True    True
        # Q1       False   False   True
        # Q2       False   False   False
        #
        #
        # Token 0 chỉ nhìn token 0
        # Token 1 nhìn token 0,1
        # Token 2 nhìn token 0,1,2
        # ============================================================
        mask_bool = (
            q_positions.unsqueeze(-1)
            <
            k_positions.unsqueeze(0)
        )

        # ============================================================
        # Những vị trí True sẽ được đổi thành -inf.
        #
        # Sau softmax:
        #
        # exp(-inf) = 0
        #
        # => token tương lai nhận attention weight = 0.
        # ============================================================
        attn_scores.masked_fill_(
            mask_bool,
            -torch.inf
        )

        # ============================================================
        # SCALED DOT-PRODUCT ATTENTION
        #
        # chia cho sqrt(head_dim)
        #
        # để attention scores không quá lớn.
        #
        # softmax theo chiều Key:
        #
        # mỗi Query phân phối xác suất attention
        # lên tất cả Key.
        #
        # Shape giữ nguyên:
        #
        # (b, heads, num_tokens_Q, num_tokens_K)
        # ============================================================
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        # ============================================================
        # WEIGHTED SUM VALUE
        #
        # attn_weights:
        # (b, heads, Q_tokens, K_tokens)
        #
        # values:
        # (b, heads, K_tokens, head_dim)
        #
        # =>
        #
        # context:
        # (b, heads, Q_tokens, head_dim)
        #
        #
        # Ví dụ Q_is:
        #
        # attention:
        #
        # I     = 0.1
        # love  = 0.2
        # AI    = 0.6
        # is    = 0.1
        #
        # context_is =
        #
        # 0.1*V_I
        # + 0.2*V_love
        # + 0.6*V_AI
        # + 0.1*V_is
        # ============================================================
        context_vec = (
            attn_weights @ values
        ).transpose(1, 2)

        # Sau transpose:
        #
        # (b, Q_tokens, heads, head_dim)
        #
        # Ví dụ:
        #
        # (1, 1, 12, 64)

        # ============================================================
        # GHÉP CÁC HEAD LẠI
        #
        # heads * head_dim = d_out
        #
        # 12 * 64 = 768
        #
        # (b, Q_tokens, 12, 64)
        #
        # ->
        #
        # (b, Q_tokens, 768)
        # ============================================================
        context_vec = context_vec.contiguous().view(
            b,
            num_tokens,
            self.d_out
        )

        # ============================================================
        # OUTPUT PROJECTION
        #
        # Cho phép model trộn thông tin giữa các head.
        #
        # shape không đổi:
        #
        # (b, num_tokens, d_out)
        # ->
        # (b, num_tokens, d_out)
        # ============================================================
        context_vec = self.out_proj(context_vec)

        return context_vec

    def reset_cache(self):

        # Xóa toàn bộ lịch sử K/V.
        #
        # Gọi khi:
        # - bắt đầu prompt/request mới
        # - kết thúc generation cũ
        self.cache_k, self.cache_v = None, None
        self.ptr_current_pos = 0

In [32]:
import torch

torch.manual_seed(42)

# ============================================================
# Giả sử câu:
# "I love AI"
#
# token 0 = "I"
# token 1 = "love"
# token 2 = "AI"
#
# Mỗi token được biểu diễn bằng vector 8 chiều.
# batch_size = 1 vì chỉ có 1 câu.
# seq_len = 3 vì có 3 token.
# d_in = 8 vì mỗi token có 8 feature.
# ============================================================

x = torch.randn(1, 3, 8)

mha = MultiHeadAttention(
    d_in=8,
    d_out=8,
    dropout=0.0,
    num_heads=2
)

mha.eval()

# Vì:
# d_out = 8
# num_heads = 2
#
# => head_dim = 8 / 2 = 4
#
# Mỗi token sau Wk/Wv sẽ từ:
# (8 feature)
#
# tách thành:
# head 0: 4 feature
# head 1: 4 feature


# ============================================================
# Decode từng token giống lúc LLM generate
# ============================================================

mha.reset_cache()

tokens = ["I", "love", "AI"]

for i in range(3):

    # Lấy đúng 1 token:
    #
    # x_i.shape = (1, 1, 8)
    #
    # 1 đầu tiên = batch size
    # 1 thứ hai   = đang xử lý 1 token
    # 8           = vector embedding/token có 8 chiều
    x_i = x[:, i:i+1, :]
    out = mha(x_i, use_cache=True)

    print(f"\nToken {i}: '{tokens[i]}'")

    # cache_k có shape:
    #
    # (batch, số_token_đã_cache, num_heads, head_dim)
    #
    # Ví dụ token đầu:
    # (1, 1, 2, 4)
    #
    # nghĩa là:
    # 1 câu
    # 1 token đã lưu
    # 2 attention heads
    # mỗi head có K vector 4 chiều
    print("K cache:", mha.cache_k.shape)
    print("V cache:", mha.cache_v.shape)

x shape: torch.Size([1, 1, 8])

Token 0: 'I'
K cache: torch.Size([1, 1, 2, 4])
V cache: torch.Size([1, 1, 2, 4])
x shape: torch.Size([1, 1, 8])

Token 1: 'love'
K cache: torch.Size([1, 2, 2, 4])
V cache: torch.Size([1, 2, 2, 4])
x shape: torch.Size([1, 1, 8])

Token 2: 'AI'
K cache: torch.Size([1, 3, 2, 4])
V cache: torch.Size([1, 3, 2, 4])


In [33]:
import torch
import torch.nn as nn
#####################################
# Chapter 4
#####################################
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.emb_dim = emb_dim
        self.weight = nn.Parameter(torch.ones(emb_dim)).float()

    def forward(self, x):
        means = x.pow(2).mean(dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(means + self.eps)
        return (x_normed * self.weight).to(dtype=x.dtype)


In [34]:

torch.manual_seed(123)

example_batch = torch.randn(2, 3, 4)

rms_norm = RMSNorm(emb_dim=example_batch.shape[-1])
rmsnorm_pytorch = torch.nn.RMSNorm(example_batch.shape[-1], eps=1e-5)

assert torch.allclose(rms_norm(example_batch), rmsnorm_pytorch(example_batch))

In [35]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

# class FeedForward(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()
#         self.layers = nn.Sequential(
#             nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
#             GELU(),
#             nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
#         )

#     def forward(self, x):
#         return self.layers(x)


In [36]:
class SiLU(nn.Module):
    def __init__(self):
        super(SiLU, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)
silu = SiLU()

assert torch.allclose(silu(example_batch), torch.nn.functional.silu(example_batch))


# Uses SwiGLU instead of GeLU to make it more comparable to MoE
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.value_proj = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.out_proj = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)
        self.silu = SiLU()

    def forward(self, x):
        x_gate = self.gate_proj(x)
        x_value = self.value_proj(x)
        x = self.silu(x_gate) * x_value
        return self.out_proj(x)

In [37]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, use_cache=False):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)

        # x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        ####################################################
        #  KV cache-related
        x = self.att(x, use_cache=use_cache)
        ####################################################

        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        use_cuda = torch.cuda.is_available()
        if use_cuda:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            base_mem = torch.cuda.memory_allocated()
        start = time.perf_counter()
        x = self.ff(x)
        if use_cuda:
            torch.cuda.synchronize()
            peak_mem = torch.cuda.max_memory_allocated()
            FFN_MEM_BYTES.append(peak_mem - base_mem)
        FFN_TIME_MS.append((time.perf_counter() - start) * 1000.0)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

In [42]:
class GPTModel(nn.Module):
    '''Small GPT language model using the MHA/KV-cache blocks above.'''
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'], dtype=cfg['dtype'])
        self.position_embedding = nn.Embedding(cfg['context_length'], cfg['emb_dim'], dtype=cfg['dtype'])
        self.dropout = nn.Dropout(cfg['drop_rate'])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg['n_layers'])])
        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.lm_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False, dtype=cfg['dtype'])
        self.lm_head.weight = self.token_embedding.weight

    def reset_cache(self):
        for block in self.blocks:
            block.att.reset_cache()

    def forward(self, input_ids, use_cache=False):
        if input_ids.ndim != 2 or input_ids.shape[1] == 0:
            raise ValueError('input_ids must have shape (batch, tokens) and be non-empty')
        batch, tokens = input_ids.shape
        if tokens > self.cfg['context_length']:
            raise ValueError('input is longer than context_length')
        if use_cache and self.blocks[0].att.cache_k is not None:
            past_tokens = self.blocks[0].att.cache_k.shape[1]
        else:
            past_tokens = 0
        if past_tokens + tokens > self.cfg['context_length']:
            raise ValueError('cached input exceeds context_length; reset_cache() first')
        positions = torch.arange(past_tokens, past_tokens + tokens, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions).unsqueeze(0)
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        return self.lm_head(self.final_norm(x))

    @torch.inference_mode()
    def generate(self, input_ids, max_new_tokens, use_cache=True):
        output = input_ids.clone()
        self.eval()
        self.reset_cache()
        if use_cache:
            logits = self(output[:, -self.cfg['context_length']:], use_cache=True)
            for _ in range(max_new_tokens):
                next_token = logits[:, -1].argmax(dim=-1, keepdim=True)
                output = torch.cat((output, next_token), dim=1)
                if output.shape[1] >= self.cfg['context_length']:
                    self.reset_cache()
                    logits = self(output[:, -self.cfg['context_length']:], use_cache=True)
                else:
                    logits = self(next_token, use_cache=True)
        else:
            for _ in range(max_new_tokens):
                logits = self(output[:, -self.cfg['context_length']:])
                output = torch.cat((output, logits[:, -1].argmax(dim=-1, keepdim=True)), dim=1)
        return output

cfg = {
    'vocab_size': 49152,
    'context_length': 8192,

    'emb_dim': 576,
    'hidden_dim': 1536,   # ← không phải 4*576 = 2304

    'n_heads': 9,
    'n_kv_groups': 3,     # 3 Q heads share 1 KV head
                          # => n_kv_heads = 9 // 3 = 3

    'n_layers': 30,       # ← SmolLM2-135M thật là 30 layers

    'drop_rate': 0.0,
    'qkv_bias': False,

    'dtype': torch.float32,  # để học/debug thì FP32 OK
}
torch.manual_seed(123)
model = GPTModel(cfg).eval()
prompt = torch.randint(0, cfg['vocab_size'], (1, 8))
full_logits = model(prompt)
model.reset_cache()
_ = model(prompt[:, :4], use_cache=True)
cached_logits = model(prompt[:, 4:], use_cache=True)
assert torch.allclose(full_logits[:, 4:], cached_logits, atol=1e-5, rtol=1e-5)
model.reset_cache()
uncached = model.generate(prompt, max_new_tokens=4, use_cache=False)
cached = model.generate(prompt, max_new_tokens=4, use_cache=True)
assert torch.equal(uncached, cached)
print('model ready; logits:', tuple(full_logits.shape), 'generation parity: OK')

model ready; logits: (1, 8, 49152) generation parity: OK


In [43]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Model size:       {total_params / 1e6:.3f}M")

Total params:     152,557,056
Trainable params: 152,557,056
Model size:       152.557M
